# Validation: CSV Export of Results Matches SQL DB Output

In [13]:
# Enumerate the sqlite results dbs
import csv
from pathlib import Path

sqlite_db_files = list(
    Path(r"Z:\_\temp\output\imported_output\results\results_20260406_082124").glob(
        "*.db"
    )
)

comparison_columns = [
    "result_id",
    "note_id",
    "batch_group",
    "patient_id",
    "window_text",
    "note_date",
    "search_term",
    "problem_list_notes_group",
    "outpatient_notes_group",
    "communication_encounter_notes_group",
    "inpatient_notes_group",
    "admission_notes_group",
    "emergency_department_notes_group",
    "ecg_impression_notes_group",
    "discharge_summary_notes_group",
    "is_patient_communication",
    "is_note_within_visit",
    "is_negated",
    "window_start_char_offset",
    "window_end_char_offset",
    "entity_start_offset",
    "entity_end_offset",
]

results_select_query = """
SELECT
	{}
FROM
	results
""".format(
    ", ".join(comparison_columns)
)

merge_columns_for_comparison = ["note_id", "search_term", "entity_start_offset"]
excluded_columns_for_comparison = [
    "result_id"
]

# Path to CSV export of sqlite results
csv_export_path = Path(r"Z:\_\temp\output\pipeline_results.csv")

# Flag to indicate whether to continue with validation
continue_flag: bool = True

In [14]:
# 1. Validate that the number of sqlite db files matches the indexes of the sqlite db files
expected_db_count: int = len(sqlite_db_files)
max_index: int = max(int(db_file.stem.split("_")[-1]) for db_file in sqlite_db_files)
if expected_db_count != max_index + 1:
    display(
        f"Error: Expected {expected_db_count} sqlite db files, but found {max_index + 1} based on indexes."
    )
    continue_flag = False

In [15]:
# 2. Validate that the CSV export file exists
if not csv_export_path.exists():
    display(f"Error: CSV export file not found at {csv_export_path}")
    continue_flag = False

In [18]:
# 3. For each sqlite db file, validate that there is a corresponding entry in the CSV export file
import pandas as pd
import sqlite3

COMPARISON_BATCH_SIZE = 10000


def validate_noteids_same(csv_export_df, db_results_df):
    return set(db_results_df["note_id"]) - set(csv_export_df["note_id"])


def compare_columns(merged_df: pd.DataFrame, column: str, csv_export_df: pd.DataFrame, db_results_df: pd.DataFrame):
    # Using pandas compare all rows in db_results_df with corresponding rows in
    # csv_export_df based on note_id, and display note_id where different
    if column in merge_columns_for_comparison + excluded_columns_for_comparison:
        return pd.DataFrame()  # Skip comparison for these columns

    db_column: str = f"{column}_db"
    csv_column: str = f"{column}_csv"
    different_rows: pd.DataFrame = merged_df[
        merged_df[db_column] != merged_df[csv_column]
    ]

    if len(different_rows) > 0:
        display(
            f"Found {len(different_rows)} different rows for column '{column}' between DB and CSV export."
        )

    return different_rows


def create_merged_extract(csv_export_df, db_results_df):
    return db_results_df.merge(
        csv_export_df,
        on=["note_id", "search_term", "entity_start_offset"],
        suffixes=("_db", "_csv"),
    )


def create_dataframe_for_results(rows) -> pd.DataFrame:
    return pd.DataFrame(rows, columns=comparison_columns)


def compare_columns_in_csv_and_db(db_file, merged_df, column, csv_export_df, db_results_df):
    continue_flag = True
    # Column-by-column comparison to identify differences
    different_rows = compare_columns(merged_df, column, csv_export_df, db_results_df)
    if not different_rows.empty:
        display(
            f"Error: Differences found in column '{column}' for note_ids {different_rows['note_id'].tolist()} between {db_file} and CSV export."
        )
        continue_flag = False

    return continue_flag


if continue_flag:

    csv_export_df: pd.DataFrame = pd.read_csv(csv_export_path)

    display("{} rows loaded from {}.".format(len(csv_export_df), csv_export_path))

    for db_file in sqlite_db_files:

        with sqlite3.connect(db_file) as conn:
            cursor = conn.cursor()
            cursor.execute(results_select_query)

            # Fetch in batches
            while True:

                rows = cursor.fetchmany(COMPARISON_BATCH_SIZE)
                if not rows:
                    break

                display(
                    "Comparing batch of {} rows from {}...".format(len(rows), db_file)
                )

                # Create dataframe from rows, using the column names from comparison_columns
                db_results_df = create_dataframe_for_results(rows)

                # Using pandas, find all the note_ids in db_results_df that are not in csv_export_df
                missing_note_ids = validate_noteids_same(csv_export_df, db_results_df)
                if missing_note_ids:
                    display(
                        f"Error: note_ids {missing_note_ids} from {db_file} not found in CSV export."
                    )
                    continue_flag = False
                else:

                    merged_df: pd.DataFrame = create_merged_extract(
                        csv_export_df, db_results_df
                    )

                    for column in comparison_columns:
                        continue_flag: bool = compare_columns_in_csv_and_db(
                            db_file, merged_df, column, csv_export_df, db_results_df
                        )
                        if not continue_flag:
                            break

                if not continue_flag:
                    display("Validation failed. Stopping further checks.")
                    break

if not continue_flag:
    display("Validation failed. Please review the errors above and address them before re-running the validation.")
else:
    display("Validation successful. All sqlite db files match the CSV export results.")

'78530 rows loaded from Z:\\_\\temp\\output\\pipeline_results.csv.'

'Comparing batch of 10000 rows from Z:\\_\\temp\\output\\imported_output\\results\\results_20260406_082124\\results_0.db...'

'Comparing batch of 1481 rows from Z:\\_\\temp\\output\\imported_output\\results\\results_20260406_082124\\results_0.db...'

'Comparing batch of 10000 rows from Z:\\_\\temp\\output\\imported_output\\results\\results_20260406_082124\\results_1.db...'

'Comparing batch of 1699 rows from Z:\\_\\temp\\output\\imported_output\\results\\results_20260406_082124\\results_1.db...'

...

'Comparing batch of 1449 rows from Z:\\_\\temp\\output\\imported_output\\results\\results_20260406_082124\\results_6.db...'

'Comparing batch of 9139 rows from Z:\\_\\temp\\output\\imported_output\\results\\results_20260406_082124\\results_7.db...'

'Comparing batch of 10000 rows from Z:\\_\\temp\\output\\imported_output\\results\\results_20260406_082124\\results_8.db...'

'Comparing batch of 1530 rows from Z:\\_\\temp\\output\\imported_output\\results\\results_20260406_082124\\results_8.db...'

'Validation successful. All sqlite db files match the CSV export results.'